# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
import re
import tensorflow as tf
from models.llama3.generation import Llama
from tqdm import tqdm 
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [5]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

In [6]:
q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/balanced_dataset.csv')
queries = q_df['question']
label = q_df['label'].str.lower().map(label_mapper)

# Question Classifier

## Text-Classification

In [14]:
tokenizer = AutoTokenizer.from_pretrained("uw-vta/bloominzer-0.1")
model = AutoModelForSequenceClassification.from_pretrained("uw-vta/bloominzer-0.1")
bloominzer = pipeline("text-classification", model=model, tokenizer=tokenizer)

Device set to use mps:0


In [15]:
message = bloominzer("How many total disk access is needed to search a record using two level indexing?")
print(message[0]['label'])

Knowledge


### Assign Prediction Labels

In [13]:
pred_labels= []
for query in tqdm(queries):
    message = bloominzer(query)
    pred_labels.append(message[0]['label'])

100%|██████████| 600/600 [00:10<00:00, 54.79it/s]


In [14]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.93      0.43      0.59       100
           1       0.39      0.89      0.54       100
           2       0.46      0.47      0.47       100
           3       0.92      0.34      0.50       100
           4       0.80      0.40      0.53       100
           5       0.54      0.72      0.62       100

    accuracy                           0.54       600
   macro avg       0.67      0.54      0.54       600
weighted avg       0.67      0.54      0.54       600



## LLM Classification

## Zero-Shot

### BART

In [12]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

bart = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [13]:
sequence_to_classify = "How many total disk access is needed to search a record using two level indexing?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
p_label = bart(sequence_to_classify, candidate_labels)
p_label['labels'][0]

'analysis'

### Assign Prediction Label

In [24]:
pred_labels= []
for query in tqdm(queries):
    sequence_to_classify = query
    candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
    p_label = bart(sequence_to_classify, candidate_labels)
    pred_labels.append(p_label['labels'][0])

100%|██████████| 600/600 [01:15<00:00,  7.90it/s]


In [26]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.37      0.17      0.23       100
           1       0.35      0.18      0.24       100
           2       0.36      0.41      0.38       100
           3       0.30      0.63      0.41       100
           4       0.36      0.18      0.24       100
           5       0.50      0.64      0.56       100

    accuracy                           0.37       600
   macro avg       0.37      0.37      0.34       600
weighted avg       0.37      0.37      0.34       600



### mDeBERTa-v3-base-mnli-xnli

In [10]:
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ya_classifier = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [11]:
sequence_to_classify = "How many total disk access is needed to search a record using two level indexing?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = ya_classifier(sequence_to_classify, candidate_labels)
label['labels'][0]

'evaluation'

### Assign Prediction Label

In [75]:
pred_labels= []
for query in tqdm(queries):
    sequence_to_classify = query
    candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
    p_label = ya_classifier(sequence_to_classify, candidate_labels)
    pred_labels.append(p_label['labels'][0])


100%|██████████| 600/600 [01:13<00:00,  8.17it/s]


In [78]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.24      0.48      0.32       100
           1       0.48      0.13      0.20       100
           2       0.25      0.52      0.34       100
           3       0.51      0.38      0.44       100
           4       0.25      0.04      0.07       100
           5       0.61      0.45      0.52       100

    accuracy                           0.33       600
   macro avg       0.39      0.33      0.31       600
weighted avg       0.39      0.33      0.31       600



## Text Generation

### LLAMA

In [3]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="modularai/Llama-3.1-8B-Instruct-GGUF",
	filename="llama-3.1-8b-instruct-q4_k_m.gguf",
    verbose=False
)


llama_context: n_ctx_per_seq (512) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h80           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h96 

In [4]:
query = 'How many total disk access is needed to search a record using two level indexing?'

message = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": f"""
        Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

        **Bloom's Taxonomy Levels:**
        1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
        - Keywords: define, list, memorize, recall, repeat
        2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
        - Keywords: describe, explain, paraphrase, summarize
        3. Application: Using learned information in new concrete situations to solve problems
        - Keywords: apply, demonstrate, solve, use, implement
        4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
        - Keywords: analyze, compare, contrast, differentiate, examine
        5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
        - Keywords: create, design, propose, formulate, integrate
        6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
        - Keywords: assess, critique, defend, evaluate, justify

        **Classification Rules:**
        - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
        - Choose the HIGHEST level that substantially applies
        - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
        - If multiple levels apply, select the most complex

        **Query to Classify:**
        "{query}"
    """
        }
    ]
)

print(message['choices'][0]['message']['content'])

analysis


### Assign Labels

In [51]:
pred_labels= []
for query in tqdm(queries):
    message = llm.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": f"""
            Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

            **Bloom's Taxonomy Levels:**
            1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
            - Keywords: define, list, memorize, recall, repeat
            2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
            - Keywords: describe, explain, paraphrase, summarize
            3. Application: Using learned information in new concrete situations to solve problems
            - Keywords: apply, demonstrate, solve, use, implement
            4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
            - Keywords: analyze, compare, contrast, differentiate, examine
            5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
            - Keywords: create, design, propose, formulate, integrate
            6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
            - Keywords: assess, critique, defend, evaluate, justify

            **Classification Rules:**
            - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
            - Choose the HIGHEST level that substantially applies
            - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
            - If multiple levels apply, select the most complex

            **Query to Classify:**
            "{query}"
        """
            }
        ]
    )
    pred_labels.append(message['choices'][0]['message']['content'].lower())
pred_labels = [
    entry.split('\n\n')[0] if entry.split('\n\n')[0] in label_mapper.keys()
    else entry
    for entry in pred_labels
]

100%|██████████| 600/600 [13:27<00:00,  1.35s/it]


In [52]:
# Reinforce 1
def q_classifier(query, prev_res):
    message = llm.create_chat_completion(
        messages=[
        {
            "role": "user", 
            "content": f"""
            Predict Bloom's Taxonomy level of understanding. Classify query: {query} in one word.
            Response generated so far {prev_res}, You have to extract blooms taxanomy and y.
            Your entire response must be just one word chosen from following: 
            {label_mapper.keys()}
            """
        }
    ]
    )
    
    return message['choices'][0]['message']['content'].lower()

In [53]:
while(len(set(pred_labels)) != 6):
    for i , query in enumerate(queries):
        if(pred_labels[i].lower() not in label_mapper.keys()):
            prev_res = pred_labels[i]
            print(prev_res)
            pred_labels[i] = q_classifier(query, prev_res)
    print(set(pred_labels))

based on the given query, the classification is: **synthesis**

the query requires the student to combine elements (the book's content) to form a new whole (organized sections with subtitles), which is a key characteristic of the synthesis level.
analysis 

the query involves breaking down the information about the cornell method and examining its effectiveness in transforming a poor lecture into a valuable learning experience. it requires analyzing the relationships between the method and its potential outcomes, which aligns with the analysis level of bloom's taxonomy.
the query "combine any two sports to develop a new olympic sport" can be classified as **synthesis**.

the reason is that the query requires combining elements (sports) to form a new whole (a new olympic sport), which is a key characteristic of the synthesis level. this level involves proposing solutions or designing new approaches, which is exactly what the query is asking for.
analysis 

the query requires breaking do

In [55]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.70      0.82       100
           1       0.74      0.63      0.68       100
           2       0.62      0.49      0.55       100
           3       0.29      0.92      0.44       100
           4       1.00      0.12      0.21       100
           5       0.97      0.33      0.49       100

    accuracy                           0.53       600
   macro avg       0.77      0.53      0.53       600
weighted avg       0.77      0.53      0.53       600



### OWEN

In [5]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

qwen_classifier = pipeline("text-generation", model=model , tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:13<00:00,  6.56s/it]
Device set to use mps:0


In [7]:
query = 'How many total disk access is needed to search a record using two level indexing?'

messages = [
    {
        "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            '''
    }
]
message = qwen_classifier(messages)

print(message[0]['generated_text'][1]['content'])

application


### Predict Label Assignment

In [ ]:
# Reinforce 1
def q_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            REVISE YOUR CLASSIFICATION. Your previous response '{prev_res}' was INVALID. 
            Classify this query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            CRITICAL RULES:
            1. MUST select from the 6 specified terms - NO exceptions
            2. Use these precise definitions:
            - knowledge: Recalling facts, terms, basic concepts (identify, list, name)
            - comprehension: Explaining meaning (describe, discuss, summarize)
            - application: Using information in new situations (execute, implement, solve)
            - analysis: Drawing connections among ideas (differentiate, organize, attribute)
            - synthesis: Producing new patterns (design, construct, integrate)
            - evaluation: Making judgments with evidence (appraise, defend, recommend)
            3. If multiple levels apply, choose the HIGHEST appropriate level
            4. Respond ONLY with the lowercase taxonomy word - NO other text

            Query: "{query}"

            Re-evaluate carefully. Your response MUST be exactly one word from the list.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [ ]:
# Reinforce 2
def q1_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            Predict Bloom's Taxonomy level of understanding. Classify query: {query} in one word. Responding {prev_res} is danger.
            Your entire response must be just one word chosen from following: 
            {label_mapper.keys()}
            {prev_res} is incorrect.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [119]:
pred_labels= []
for query in tqdm(queries):
    messages = [
        {
            "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            ''',
        }
    ]

    message = qwen_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'][1]['content'])

while(len(set(pred_labels)) != 6):
    for i , query in enumerate(queries):
        if(pred_labels[i].lower() not in label_mapper.keys()):
            prev_res = pred_labels[i]
            print(prev_res)
            pred_labels[i] = q_classifier(query, prev_res)
            if(pred_labels[i].lower() not in label_mapper.keys()):
                pred_labels[i] = q1_classifier(query, prev_res)
    print(set(pred_labels))

100%|██████████| 600/600 [08:16<00:00,  1.21it/s]


composition
explanation
comparison
definition
definition
definition
comparison
comparison
recognition
definition
definition
definition
{'comprehension', 'synthesis', 'comparison is evaluation.', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
comparison is evaluation.
definition is incorrect.
comparison is evaluation.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledg

In [120]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.51      0.68       100
           1       0.72      0.43      0.54       100
           2       0.34      0.69      0.46       100
           3       0.60      0.79      0.68       100
           4       0.77      0.56      0.65       100
           5       0.86      0.72      0.78       100

    accuracy                           0.62       600
   macro avg       0.71      0.62      0.63       600
weighted avg       0.71      0.62      0.63       600



## Google-FLAN-T5-XL

In [8]:
model_name = "google/flan-t5-xl"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

flan_classifier = pipeline("text2text-generation", model=model , tokenizer=tokenizer, device=-1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 14.99it/s]
Device set to use cpu


In [9]:
query = 'How many total disk access is needed to search a record using two level indexing?'

messages = f"""
    Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

    **Bloom's Taxonomy Levels:**
    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
    - Keywords: define, list, memorize, recall, repeat
    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
    - Keywords: describe, explain, paraphrase, summarize
    3. Application: Using learned information in new concrete situations to solve problems
    - Keywords: apply, demonstrate, solve, use, implement
    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
    - Keywords: analyze, compare, contrast, differentiate, examine
    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
    - Keywords: create, design, propose, formulate, integrate
    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
    - Keywords: assess, critique, defend, evaluate, justify

    **Classification Rules:**
    - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
    - Choose the HIGHEST level that substantially applies
    - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
    - If multiple levels apply, select the most complex

    **Query to Classify:**
    "{query}"
        """

message = flan_classifier(messages)
print(message)

[{'generated_text': 'application'}]


### Predict Label Assignment

In [15]:
pred_labels= []
for query in tqdm(queries):
    messages = f"""
        Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

        **Bloom's Taxonomy Levels:**
        1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
        - Keywords: define, list, memorize, recall, repeat
        2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
        - Keywords: describe, explain, paraphrase, summarize
        3. Application: Using learned information in new concrete situations to solve problems
        - Keywords: apply, demonstrate, solve, use, implement
        4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
        - Keywords: analyze, compare, contrast, differentiate, examine
        5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
        - Keywords: create, design, propose, formulate, integrate
        6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
        - Keywords: assess, critique, defend, evaluate, justify

        **Classification Rules:**
        - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
        - Choose the HIGHEST level that substantially applies
        - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
        - If multiple levels apply, select the most complex

        **Query to Classify:**
        "{query}"
    """

    message = flan_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'].lower())

100%|██████████| 600/600 [3:05:12<00:00, 18.52s/it]  


In [19]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.76      0.86       100
           1       0.98      0.40      0.57       100
           2       0.41      0.68      0.51       100
           3       0.88      0.65      0.75       100
           4       0.44      0.84      0.58       100
           5       0.96      0.53      0.68       100

    accuracy                           0.64       600
   macro avg       0.78      0.64      0.66       600
weighted avg       0.78      0.64      0.66       600

